In [ ]:
import json
import urllib.error
import urllib.request
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import scipy.stats as stats
import itertools

import altair as alt
from sklearn.metrics import precision_recall_curve, auc
from natsort import natsorted

# Overhead for finding data

In [ ]:
input_directory = Path('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc')
scores_excel = Path('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/20260101_SGEsubset.xlsx')

all_scores = pd.read_excel(scores_excel)

In [ ]:
def find_genes(input_dir: Path | None = None) -> dict:
    """Discover all gene datasets in the input directory.

    Detects genes by finding all *delcounts.tsv files and extracting the gene
    name from each filename. Both dot-separated (e.g. CTCF.delcounts.tsv) and
    run-together (e.g. 20260129_RAD51Ddelcounts.tsv) naming conventions are
    supported. Companion files (*snvcounts.tsv, *editrates.tsv) must also be
    present for each detected gene. Per-gene scores are sourced from a shared
    Excel file (filtered by the 'Gene' column) passed via scores_excel.

    Args:
        input_dir: Directory containing gene-specific TSV files.
        scores_excel: Path to the shared scores Excel file (e.g.
            20260101_SGEsubset.xlsx). Scores for each gene are obtained by
            filtering rows where Gene == gene_name.

    Returns a dict mapping gene name -> files dict, e.g.:
        {"RAD51D": {"del_counts": Path(...), "snv_counts": Path(...), ...}}
    """
    delcounts_files = sorted(input_dir.glob("*delcounts.tsv"))
    if not delcounts_files:
        raise FileNotFoundError(f"No '*delcounts.tsv' files found in {input_dir}")

    def find_one(*patterns):
        for pattern in patterns:
            matches = list(input_dir.glob(pattern))
            if len(matches) == 1:
                return matches[0]
            if len(matches) > 1:
                raise ValueError(
                    f"Multiple files match '{pattern}': "
                    + ", ".join(str(m) for m in matches)
                )
        raise FileNotFoundError(
            f"Could not find any of {patterns} in {input_dir}"
        )

    genes = {}
    for delcounts_path in delcounts_files:
        # Handles both GENE.delcounts.tsv and GENEdelcounts.tsv (with optional prefix)
        stem_part = delcounts_path.stem.split("_")[-1]
        gene = stem_part.removesuffix(".delcounts").removesuffix("delcounts")

        gene_score_df = all_scores.loc[all_scores['Gene'] == gene].copy()
        genes[gene] = {
            "del_counts": delcounts_path,
            "snv_counts": find_one(f"*{gene}snvcounts.tsv", f"*{gene}.snvcounts.tsv"),
            "edit_rates": find_one(f"*{gene}editrates.tsv", f"*{gene}.editrates.tsv"),
            "scores_excel": gene_score_df
        }

    return genes

In [ ]:
gene_paths= find_genes(input_directory)

# Helper functions to build visualizations

In [ ]:
#Read all data into dictionary of dataframes
def read_data(gene):
    data_dict = {}
    paths = gene_paths[gene]

    to_read = ['del_counts', 'snv_counts', 'edit_rates']

    for elem in to_read:
        df = pd.read_csv(paths[elem], sep = '\t')

        if elem=='edit_rates':

            def recode_reps(df):

                if len(df) != 3:
                    df = df.assign(rep=['R1', 'R2'])

                return df
            
            df['target'] = df['target_rep'].transform(lambda x: x.split('_')[1].split('X')[1])
            df['rep'] = df['target_rep'].transform(lambda x: x.split('_')[2])

            rep_map = {'R1R4': 'R1',
                       'R1R2R3': 'R1',
                       'R2R5': 'R2',
                       'R4R5R6': 'R2',
                       'R3R6': 'R3',
                       'R7R8R9': 'R3'
                       }
            
            df['rep'] = df['rep'].map(rep_map)

            df = df.groupby('target').apply(recode_reps).reset_index(drop = True)
        
        if elem=='snv_counts':
            day_columns = [col for col in df.columns.tolist() if "D" in col]
            new_day_columns = [col.replace('_', ' ') for col in day_columns]
            
            new_col_names = dict(zip(day_columns, new_day_columns))

            df = df.rename(columns = new_col_names)
            
            df['max_count'] = df[new_day_columns].max(axis = 1)
            df = df.loc[df['max_count'] < 100000].copy()

        data_dict[elem] = df
    
    data_dict['scores'] = paths['scores_excel']
    
    return data_dict

In [ ]:
#Editing rate bar plot
def edit_rate_barplot(df):
    sort_order = natsorted(df['target'].tolist())
    plot = alt.Chart(df).mark_bar().encode(
        x = alt.X('rep',
                  axis = alt.Axis(
                      title = '',
                      labels = False,
                      ticks = False
                  )
                 ),
        y = alt.Y('edit_rate',
                  title = 'Lib. Edit Rate',
                  axis = alt.Axis(labelFontSize = 14,
                                  titleFontSize = 16
                  ),
                  scale = alt.Scale(domain = [0, 0.5])
                 ),
        column = alt.Column('target',
                            sort = sort_order,
                            header=alt.Header(title='SGE Target')
                           ),
        color = alt.Color('rep', 
                          legend = alt.Legend(
                              title = '',
                              titleFontSize = 16,
                              labelFontSize = 14,
                              labelFont = 'Arial',
                              titleFont = 'Arial',
                              orient = 'bottom'
                          )
                         )
    ).properties(
        width = 25,
        height = 200
    ).configure_facet(
        spacing = 5
    ).configure_axis(
        grid = False,
        labelFont = 'Arial',
        titleFont = 'Arial'
    ).configure_header(
        labelFontSize = 14,
        labelFont = 'Arial',
        titleFont = 'Arial'
    )

    plot.display()

    return plot

In [ ]:
# Correlation heatmap

def corr_heatmap(df):
    target_dfs = df.groupby('target')

    final_tuples = []
    for target, df in target_dfs:
        all_day_cols = [col for col in df.columns.tolist() if "D" in col]

        d05_cols = [col for col in all_day_cols if "D05" in col]
        d13_cols = [col for col in all_day_cols if "D13" in col]

        d17_cols = []
        if "D17 R1" in all_day_cols:
            d17_cols = [col for col in all_day_cols if "D17" in col]
            day_cols = [d05_cols, d13_cols, d17_cols]
        else:
            day_cols = [d05_cols, d13_cols]

        test_pairs_dict = {}
        for i, col_list in enumerate(day_cols):
            test_pairs = list(itertools.combinations(col_list, 2))

            day_map = {0: 'D5',
                       1: 'D13',
                       2: 'D17'}
            
            test_pairs_dict[day_map[i]] = test_pairs

        for day in test_pairs_dict.keys():
            tests_to_do = test_pairs_dict[day]

            for  rep1, rep2 in tests_to_do:
                if len(df.dropna(subset = [rep1, rep2])) == 0:
                    continue
                
                corr,_=stats.pearsonr(df[rep1], df[rep2])

                mod_target = target.split('_')[1].split('X')[1]
                data_tuple = (mod_target, f"{rep1} vs. {rep2.split(' ')[1]}", corr)
                final_tuples.append(data_tuple)
    
    corr_df = pd.DataFrame(final_tuples, columns = ['Targets', 'Tests', 'corr'])
    
    targets = set(corr_df['Targets'].tolist())
    targets = natsorted(targets)

    num_targets = len(targets)
    num_tests = len(corr_df['Tests'].unique())

    base = alt.Chart(corr_df, title = alt.TitleParams(text = f' Correlation of Replicates', fontSize = 32)).encode(
        x = alt.X('Tests:N'),
        y = alt.Y('Targets:N', sort = targets)
    )
    
    graph = base.mark_rect().encode(
                x = alt.X('Tests:N', axis = alt.Axis(title = '', titleFontSize = 28, labelFontSize = 24, labelLimit = 300, labelAngle = 45)),
                y = alt.Y('Targets', axis = alt.Axis(title = 'SGE Target', titleFontSize = 28, labelFontSize = 24), sort = targets),
                color = alt.Color('corr:Q', scale = alt.Scale(domain = [.2, 1]), legend = alt.Legend(title = "Pearson's r", titleFontSize = 24,labelFontSize = 22, labelFont = 'Arial', titleFont = 'Arial')),
                tooltip = [alt.Tooltip('corr', title = "Pearson's r: ")]
    ).properties(
        width = 70 * num_tests,
        height = 25 * num_targets
    )

    color = (
        alt.when(alt.datum.corr > 0.5)
        .then(alt.value("white"))
        .otherwise(alt.value("black"))
    )

    text = base.mark_text(baseline = 'middle', fontSize = 20).encode(
        text = alt.Text('corr:Q',format = "0.3f"), color = color
    ).transform_filter(
    'isValid(datum.corr)'
    )

    graph = (graph + text).configure_axis(
        grid = False,
        labelFont = 'Arial',
        titleFont = 'Arial'
    ).configure_view(
        stroke = None
    )
    
    graph.display()

    return graph




# Builds standard QC viz. for each gene

In [ ]:
for gene in gene_paths:
    print(f'This is the output for {gene}')
    data_dict = read_data(gene)

    edit_rate_plot = edit_rate_barplot(data_dict['edit_rates'])
    pearson_heatmap = corr_heatmap(data_dict['snv_counts'])